# 04 — Ekspor Dataset untuk Labeling Sentimen

Notebook ini mengekspor dataset gabungan (semua video) ke format siap labeling.
Kolom `label` dibiarkan kosong untuk diisi pada tahap anotasi.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

from configs.config import Config
Config.paths.ensure_dirs()


In [ ]:
from src.preprocess_spark import build_spark_session

spark = build_spark_session(
    app_name=f"{Config.spark.APP_NAME}_export",
    master=Config.spark.MASTER,
)


In [ ]:
parquet_path = str(Config.paths.DATA_PROCESSED / "merged_preprocessed")
df = spark.read.parquet(parquet_path)

print(f"Total baris: {df.count():,}")
df.printSchema()


In [ ]:
from pyspark.sql import functions as F

# Statistik per video
print("Distribusi data per video_id:")
df.groupBy("video_id").agg(
    F.count("comment_id").alias("total_komentar"),
    F.avg(F.length("text_svm")).alias("rata_panjang_svm"),
    F.avg(F.length("text_bert")).alias("rata_panjang_bert"),
).orderBy(F.desc("total_komentar")).show(30, truncate=False)


In [ ]:
# Buat dataset labeling dengan skema bersih
df_labeling = df.select(
    "comment_id",
    "video_id",
    "text_original",
    "text_svm",
    "text_bert",
    F.lit(None).cast("string").alias("label"),
).filter(
    (F.length(F.col("text_svm")) > 0) | (F.length(F.col("text_bert")) > 0)
)

print(f"Dataset labeling: {df_labeling.count():,} baris siap anotasi")
df_labeling.show(5, truncate=70)


In [ ]:
outputs_dir = str(Config.paths.OUTPUTS)

# JSONL
jsonl_path = os.path.join(outputs_dir, "labeling_ready_jsonl")
df_labeling.coalesce(1).write.mode("overwrite").json(jsonl_path)
print(f"[OK] JSONL: {jsonl_path}")

# CSV
csv_path = os.path.join(outputs_dir, "labeling_ready_csv")
df_labeling.coalesce(1).write.mode("overwrite").option("header", "true").csv(csv_path)
print(f"[OK] CSV : {csv_path}")

# Parquet
parquet_out = os.path.join(outputs_dir, "labeling_ready_parquet")
df_labeling.write.mode("overwrite").parquet(parquet_out)
print(f"[OK] Parquet: {parquet_out}")


In [ ]:
# Statistik akhir
print("\n=== STATISTIK AKHIR DATASET LABELING ===")
df_labeling.agg(
    F.count("comment_id").alias("total_komentar"),
    F.countDistinct("video_id").alias("jumlah_video"),
    F.avg(F.length("text_svm")).alias("rata_panjang_svm"),
    F.avg(F.length("text_bert")).alias("rata_panjang_bert"),
).show()


## Skema Output Labeling

```json
{
  "comment_id": "UgxABC123_xyz",
  "video_id": "dQw4w9WgXcQ",
  "text_original": "Mantap banget videonya bro!!",
  "text_svm": "mantap video",
  "text_bert": "mantap banget videonya bro",
  "label": null
}
```

Field `label` diisi dengan: `positif`, `negatif`, atau `netral`.

## Langkah Selanjutnya

1. Ekspor CSV/JSONL ke tool labeling (Label Studio, Doccano, atau spreadsheet).
2. Lakukan anotasi sentimen.
3. Gunakan hasil berlabel untuk:
   - Pelatihan model SVM menggunakan `text_svm` + TF-IDF
   - Fine-tuning IndoBERT menggunakan `text_bert`